<a href="https://colab.research.google.com/github/LeonAmbroseJr/-LeonAmbrose-github.io/blob/main/cheatgrass2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
# Removed geopandas import since it's not available
# from shapely.geometry import LineString

# Create sample data to replace the undefined variables
# 1. Sample habitat suitability matrix (2D array for heatmap)
habitat_suitability_matrix = np.random.rand(20, 20)  # 20x20 matrix with random values 0-1

# 2. Sample road network data (using simple coordinates instead of GeoDataFrame)
# Creating simple line coordinates to represent roads without geopandas
road_data = [
    {'x': [2, 8], 'y': [5, 12], 'road_id': 1},
    {'x': [1, 18], 'y': [15, 18], 'road_id': 2},
    {'x': [10, 15], 'y': [2, 19], 'road_id': 3}
]

# 3. Sample GBIF presence points (longitude and latitude coordinates)
np.random.seed(42)  # For reproducible random data
gbif_lon = np.random.uniform(0, 20, 15)  # 15 random longitude points
gbif_lat = np.random.uniform(0, 20, 15)  # 15 random latitude points

# Modified plotting code without geopandas
fig, ax = plt.subplots(figsize=(10, 8))

# 1. Background heatmap for PRISM Climate/Habitat Suitability (Continuous Palette)
# Using 'YlOrRd' (Yellow-Orange-Red) to indicate suitability intensity
sns.heatmap(habitat_suitability_matrix, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Invasion Suitability'})

# 2. Plotting the road network disturbance zones using matplotlib instead of geopandas
# Plot each road as a line using matplotlib's plot function
for road in road_data:
    ax.plot(road['x'], road['y'], color='black', linewidth=1.5, label='Disturbance Buffers (Roads)' if road['road_id'] == 1 else "")

# 3. Plotting ground-truth presence points
# Bright magenta or cyan dots will pop cleanly against a Yellow-Orange-Red background
plt.scatter(gbif_lon, gbif_lat, color='magenta', s=10, label='GBIF Verified Presence')

plt.title("Cheatgrass Containment Mapping: Suitability vs. Infrastructure")
plt.legend()
plt.show()

import ee
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. INITIALIZE EARTH ENGINE
# ==========================================
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

# ==========================================
# 2. DEFINE BOUNDARIES & CONFIGURATION
# ==========================================
# Extract the Pine Ridge Reservation boundary using Census TIGER data
reservations = ee.FeatureCollection("TIGER/2018/LandAndWater/AmericanIndianAreaRCRSA")
pine_ridge = reservations.filter(ee.Filter.eq('NAME', 'Pine Ridge'))

# Parameters
START_YEAR = 2015
END_YEAR = 2025
INVADED_THRESHOLD = 15  # Percent canopy cover to count a pixel as "highly invaded"
YEARS = list(range(START_YEAR, END_YEAR + 1))

# Load the RAP Vegetation Cover ImageCollection (V3 or latest)
# 'AFG' stands for Annual Forbs and Grasses (the functional group containing Cheatgrass)
rap_collection = ee.ImageCollection("projects/rap-data-365417/assets/vegetation-cover-v2")

# ==========================================
# 3. ANALYSIS PIPELINE
# ==========================================
data_summary = []

print("Analyzing Earth Engine satellite data layers...")

for year in YEARS:
    # Filter dataset for specific year and select the AFG band
    rap_year = rap_collection.filter(ee.Filter.calendarRange(year, year, 'year')).first()
    afg_cover = rap_year.select('AFG').clip(pine_ridge)

    # Calculate Mean Cover across the entire reservation
    mean_dict = afg_cover.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=pine_ridge.geometry(),
        scale=30,
        maxPixels=1e9
    )
    mean_cover = mean_dict.get('AFG').getInfo()

    # Calculate total area exceeding the invasion threshold (>15% cover)
    invaded_mask = afg_cover.gte(INVADED_THRESHOLD)
    area_image = invaded_mask.multiply(ee.Image.pixelArea())  # Area in square meters

    area_dict = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=pine_ridge.geometry(),
        scale=30,
        maxPixels=1e9
    )
    # Convert square meters to acres (1 sq meter = 0.000247105 acres)
    invaded_acres = area_dict.get('AFG').getInfo() * 0.000247105

    data_summary.append({
        'Year': year,
        'Mean_Cover_Pct': mean_cover,
        'Invaded_Acres': invaded_acres
    })
    print(f"✔ Processed Year: {year}")

# Convert findings to Pandas DataFrame
df_trends = pd.DataFrame(data_summary)

# ==========================================
# 4. DATA VISUALIZATION
# ==========================================
sns.set_theme(style="whitegrid")
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot 1: Mean Cover Trend (Line)
color = '#d95f02'
ax1.set_xlabel('Year', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean AFG Cover (%)', color=color, fontsize=12, fontweight='bold')
line = ax1.plot(df_trends['Year'], df_trends['Mean_Cover_Pct'], color=color, marker='o', linewidth=2.5, label='Mean Cover %')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(YEARS)

# Plot 2: Total Invaded Acreage (Bars)
ax2 = ax1.twinx()
color = '#7570b3'
ax2.set_ylabel(f'Total Acres Exceeding {INVADED_THRESHOLD}% Cover', color=color, fontsize=12, fontweight='bold')
bars = ax2.bar(df_trends['Year'], df_trends['Invaded_Acres'], color=color, alpha=0.3, width=0.4, label='Invaded Acreage')
ax2.tick_params(axis='y', labelcolor=color)

plt.title(f"Pine Ridge Reservation: Cheatgrass (AFG) Spread Trends ({START_YEAR}-{END_YEAR})", fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.show()

# Display the raw statistical metrics
print("\n--- Summary Statistics Table ---")
print(df_trends.to_string(index=False))

# ==========================================
# 1. INITIALIZE EARTH ENGINE
# ==========================================
import ee
try:
    # !!! REPLACE 'your-gcp-project-id' with your actual Google Cloud Project Name/ID !!!
    ee.Initialize(project='your-gcp-project-id')
except Exception as e:
    print("Authentication required or project missing. Requesting access token...")
    ee.Authenticate()
    # !!! REPLACE HERE AS WELL !!!
    ee.Initialize(project='your-gcp-project-id')

# ==========================================
# 2. DEFINE BOUNDARIES & CONFIGURATION
# ==========================================
# Extract the Pine Ridge Reservation boundary using Census TIGER data
reservations = ee.FeatureCollection("TIGER/2018/LandAndWater/AmericanIndianAreaRCRSA")
pine_ridge = reservations.filter(ee.Filter.eq('NAME', 'Pine Ridge'))

# Parameters
START_YEAR = 2015
END_YEAR = 2025
INVADED_THRESHOLD = 15  # Percent canopy cover to count a pixel as "highly invaded"
YEARS = list(range(START_YEAR, END_YEAR + 1))

# FIXED: Swapped out the legacy 'rap-data-365417' path for the official V3 collection
rap_collection = ee.ImageCollection("projects/rangeland-analysis-platform/assets/vegetation-cover-v3")

import os
import glob
import numpy as np
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt

# =========================================================================
# CONFIGURATION & PARAMETERS
# =========================================================================
# Folder containing your AppEEARS unpacked MODIS NDVI GeoTIFF rasters
TIFF_DIR = "path_to_your_appeears_outputs"

# Define DOY (Day of Year) files for your target phenological windows
# Early Spring: Cheatgrass rapid green-up (e.g., late March/April, DOY ~081 to 121)
# Early Summer: Cheatgrass senesces/turns brown while perennials green up (DOY ~161+)
SPRING_TIFF = os.path.join(TIFF_DIR, "*A2025097*.tif") # Example: DOY 097 (April 7)
SUMMER_TIFF = os.path.join(TIFF_DIR, "*A2025161*.tif") # Example: DOY 161 (June 10)

def load_and_scale_ndvi(file_pattern):
    """Finds a file matching the pattern, reads it, and applies MODIS scaling factors."""
    match = glob.glob(file_pattern)
    if not match:
        raise FileNotFoundError(f"No file found matching pattern: {file_pattern}")
    
    with rasterio.open(match[0]) as src:
        # Read first band
        ndvi = src.read(1).astype(np.float32)
        # MODIS NDVI valid range is -2000 to 10000; Scale factor is 0.0001
        ndvi = np.where((ndvi >= -2000) & (ndvi <= 10000), ndvi * 0.0001, np.nan)
        profile = src.profile
    return ndvi, profile

# =========================================================================
# PHENOLOGICAL ANALYSIS PIPELINE
# =========================================================================
print("Loading AppEEARS foundational satellite layers...")
spring_ndvi, img_profile = load_and_scale_ndvi(SPRING_TIFF)
summer_ndvi, _ = load_and_scale_ndvi(SUMMER_TIFF)

# 1. Calculate Phenological Cheatgrass Proxy Index
# Cheatgrass reaches high NDVI early, then craters dramatically as it dries out (senesces)
# While native perennials maintain or increase their NDVI during early summer.
pheno_diff = spring_ndvi - summer_ndvi

# 2. Apply Threshold Filters
# - High spring productivity (NDVI > 0.35)
# - Significant drop into summer (Difference > 0.15)
cheatgrass_mask = (spring_ndvi > 0.35) & (pheno_diff > 0.15)

# Clear up nodata values
cheatgrass_mask = np.where(np.isnan(spring_ndvi) | np.isnan(summer_ndvi), 0, cheatgrass_mask)

# =========================================================================
# EXPORT & VISUALIZATION
# =========================================================================
# Update metadata profile to output a binary mask raster (1 = Cheatgrass, 0 = Other)
img_profile.update(dtype=rasterio.uint8, count=1, nodata=0)

output_mask_path = os.path.join(TIFF_DIR, "cheatgrass_pheno_proxy.tif")
with rasterio.open(output_mask_path, "w", **img_profile) as dst:
    dst.write(cheatgrass_mask.astype(rasterio.uint8), 1)
print(f"✔ Phenological proxy raster exported to: {output_mask_path}")

# Plotting the raw signature mapping
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left plot: The phenological difference raw value
im1 = ax1.imshow(pheno_diff, cmap="bwr", vmin=-0.4, vmax=0.4)
fig.colorbar(im1, ax=ax1, label="Spring NDVI minus Summer NDVI")
ax1.set_title("Phenological Shift (Positive = Early Peak)")

# Right plot: The isolated proxy footprint
ax2.imshow(cheatgrass_mask, cmap="Greens")
ax2.set_title("Surrogated Cheatgrass Footprint (Threshold Matrix)")

plt.tight_layout()
plt.show()

Key Logic Behind This Framework:MODIS Data Clean-up: AppEEARS drops the raw HDF datasets into clean GeoTIFFs, but they are still scaled by $10,000$. The script handles the 0.0001 scale transformation automatically to give you true baseline NDVI values between $-1.0$ and $1.0$.Phenological Core Matrix: By calculating $NDVI_{Spring} - NDVI_{Summer}$, you capture the exact ecological niche of cheatgrass. Native vegetation or perennial rangeland crops will return negative or near-zero changes over this specific spring-to-summer transition window.
